## Configuration

Choose which model to run throughout this notebook: `"97m"` for the compact 97M model or `"311m"` for the 311M flagship. The Matryoshka section (section 2) requires `"311m"`.

In [ ]:
# Choose "311m" or "97m". The Matryoshka section requires "311m".
# MODEL_SIZE = "97m"
MODEL_SIZE = "311m"

MODEL_PATH = {
    "311m": "ibm-granite/granite-embedding-311m-multilingual-r2",
    "97m":  "ibm-granite/granite-embedding-97m-multilingual-r2",
}[MODEL_SIZE]

print(f"Using {MODEL_PATH}")

# --- Model cache ---
import torch

_loaded_model = None
_loaded_model_name = None

def load_model(model_name=MODEL_PATH,
               model_kwargs=None,
               config_kwargs=None,
               **kwargs):
    """Load a SentenceTransformer model, reusing the cached instance if the name matches."""
    global _loaded_model, _loaded_model_name
    import torch
    if _loaded_model is not None and _loaded_model_name == model_name:
        return _loaded_model
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if model_kwargs is None and device == "cuda":
        model_kwargs = {'dtype': torch.bfloat16, "attn_implementation": "flash_attention_2"}
    # if config_kwargs is None:
    #     config_kwargs = {'reference_compile': False}
    from sentence_transformers import SentenceTransformer
    _loaded_model = SentenceTransformer(model_name,
                                        model_kwargs=model_kwargs,
                                        # config_kwargs=config_kwargs,
                                        **kwargs)
    _loaded_model_name = model_name
    return _loaded_model

## 1. Cross-Lingual Retrieval

Encode three queries in English, German, and Japanese against their matching passages. The cosine-similarity matrix diagonal should be highest — each query retrieves the right passage across language boundaries.

### Setup

Install the dependencies (uncomment as needed). `flash_attn` is optional — it speeds up encoding on supported GPUs.

In [ ]:
!pip install sentence-transformers transformers torch

In [ ]:
# Run this if you have a GPU and want the model to run faster.
!pip install flash_attn

In [ ]:
from sentence_transformers import util

model = load_model(MODEL_PATH)

input_queries = [
    "What is the tallest mountain in Japan?",          # English query
    "Wer hat das Lied Achy Breaky Heart geschrieben?", # German query
    "ドイツの首都はどこですか？",                            # Japanese query
]

input_passages = [
    "富士山は、静岡県と山梨県にまたがる活火山で、標高3776.12 mで日本最高峰の独立峰である。",  # Japanese passage
    "Achy Breaky Heart is a country song written by Don Von Tress. Originally titled Don't Tell My Heart and performed by The Marcy Brothers in 1991.",  # English passage
    "Berlin ist die Hauptstadt und ein Land der Bundesrepublik Deutschland. Die Stadt ist mit rund 3,7 Millionen Einwohnern die bevölkerungsreichste Kommune Deutschlands.",  # German passage
]

query_embeddings = model.encode(input_queries)
passage_embeddings = model.encode(input_passages)

print(util.cos_sim(query_embeddings, passage_embeddings))
# Highest scores along the diagonal (EN→JA, DE→EN, JA→DE)

### Semantic Search

Rank a multilingual document corpus against an English query. The model retrieves the most relevant documents regardless of what language they are written in.

In [ ]:
from sentence_transformers import util

model = load_model(MODEL_PATH)

query = "What is the capital of Germany?"

documents = [
    "Berlin ist die Hauptstadt und ein Land der Bundesrepublik Deutschland.",  # German
    "富士山は、静岡県と山梨県にまたがる活火山で、標高3776.12 mで日本最高峰の独立峰である。",        # Japanese
    "Achy Breaky Heart is a country song written by Don Von Tress.",           # English
    "La Tour Eiffel est une tour en treillis de fer puddlé à Paris.",           # French
    "Germany is a federal republic in Central Europe with 84 million people.",  # English
]

query_embedding = model.encode(query)
doc_embeddings  = model.encode(documents)

scores = util.cos_sim(query_embedding, doc_embeddings)[0]
ranked = sorted(zip(scores.tolist(), documents), reverse=True)

print(f"Query: {query}\n")
for score, doc in ranked:
    print(f"{score:.3f}  {doc[:80]}")
# 0.85+  Berlin ist die Hauptstadt …
# 0.75+  Germany is a federal republic …
# lower  unrelated passages

## 2. Matryoshka Embeddings (311M only)

The 311M model is trained with [Matryoshka Representation Learning](https://arxiv.org/abs/2205.13147), so its 768-dim embeddings can be truncated to 512, 384, 256, or 128 dimensions with graceful quality loss. The 97M model does not support this — its 384-dim output is already compact.

Skip this cell if `MODEL_SIZE = "97m"`.

In [ ]:
if MODEL_SIZE == "311m":
    model = load_model(MODEL_PATH)

    full_embeddings = model.encode(["example text"])
    print(full_embeddings.shape)  # (1, 768)

    truncated_embeddings = model.encode(["example text"], truncate_dim=256)
    print(truncated_embeddings.shape)  # (1, 256)
else:
    print(f"Matryoshka is not supported by {MODEL_PATH}; switch MODEL_SIZE to '311m' to run this example.")

## 3. Raw Hugging Face Transformers

If you'd rather skip `sentence-transformers`, you can encode directly with `transformers`. Both R2 models use **CLS pooling** followed by L2 normalization.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
device="cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    model = AutoModel.from_pretrained(
        MODEL_PATH,
        # reference_compile=False,
        attn_implementation="flash_attention_2"
    ).to(device)
else:
    model = AutoModel.from_pretrained(MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model.eval()

input_queries = [
    "What is the tallest mountain in Japan?",
    "Wer hat das Lied Achy Breaky Heart geschrieben?",
    "ドイツの首都はどこですか？",
]

tokenized_queries = tokenizer(input_queries, padding=True, truncation=True, return_tensors="pt").to(device)

with torch.no_grad():
    model_output = model(**tokenized_queries)
    # CLS pooling
    query_embeddings = model_output[0][:, 0]

query_embeddings = torch.nn.functional.normalize(query_embeddings, dim=1)
print(query_embeddings.shape)

### LangChain

Use the model as a `HuggingFaceEmbeddings` object — a drop-in replacement wherever LangChain accepts an `Embeddings` instance.

In [ ]:
!pip install langchain-huggingface

In [ ]:
# LangChain — pip install langchain-huggingface
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name=MODEL_PATH)

doc_texts = [
    "富士山は日本最高峰の独立峰です。",
    "Mount Fuji is Japan's highest peak.",
]
docs = embeddings.embed_documents(doc_texts)
query = embeddings.embed_query("What is Japan's tallest mountain?")
print(f"doc embeddings:      {len(docs)} × {len(docs[0])}")
print(f"query embedding dim: {len(query)}")

# Search: cosine similarity between query and each document
docs_arr = np.array(docs)
query_arr = np.array(query)
scores = docs_arr @ query_arr  # vectors are already L2-normalised
best = int(np.argmax(scores))
for i, (score, text) in enumerate(zip(scores, doc_texts)):
    marker = " ◀ best match" if i == best else ""
    print(f"{score:.4f}  {text}{marker}")
# Drop-in replacement anywhere LangChain accepts an Embeddings object


### LlamaIndex

Install the LlamaIndex HuggingFace embeddings package, then set the model as the global embedding backend.

In [ ]:
!pip install llama-index-embeddings-huggingface

Set the model as the global embedding backend via `Settings.embed_model`; all subsequent LlamaIndex indexes and pipelines use it automatically.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

embed_model = HuggingFaceEmbedding(model_name=MODEL_PATH)
Settings.embed_model = embed_model  # applies globally to any index or pipeline

embedding = embed_model.get_text_embedding("What is Japan's tallest mountain?")
print(f"embedding dim: {len(embedding)}")

### Haystack

Install Haystack with sentence-transformers support, then use the document and query embedders in a full pipeline.

In [ ]:
# Haystack — install if needed
!pip install sentence-transformers haystack-ai

Use `SentenceTransformersDocumentEmbedder` and `SentenceTransformersTextEmbedder` to build a full embed-and-retrieve pipeline with an in-memory document store.

In [ ]:
from haystack.components.embedders import (
    SentenceTransformersDocumentEmbedder,
    SentenceTransformersTextEmbedder,
)
from haystack.dataclasses import Document
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.dataclasses import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore

doc_embedder = SentenceTransformersDocumentEmbedder(model=MODEL_PATH)
query_embedder = SentenceTransformersTextEmbedder(model=MODEL_PATH)
doc_embedder.warm_up()
query_embedder.warm_up()
doc_embedder.warm_up()
query_embedder.warm_up()

# Embed and index documents
document_store = InMemoryDocumentStore()
result_docs = doc_embedder.run(documents=[
    Document(content="富士山は日本最高峰の独立峰です。"),
    Document(content="Mount Fuji is Japan's highest peak."),
    Document(content="Achy Breaky Heart is a country song written by Don Von Tress."),
    Document(content="Berlin ist die Hauptstadt und ein Land der Bundesrepublik Deutschland."),
])
document_store.write_documents(result_docs["documents"])

# Embed query and retrieve
result_query = query_embedder.run(text="What is Japan's tallest mountain?")
retriever = InMemoryEmbeddingRetriever(document_store=document_store)
results = retriever.run(query_embedding=result_query["embedding"], top_k=2)
for doc in results["documents"]:
    print(f"{doc.score:.3f}  {doc.content}")
# 0.973  Mount Fuji is Japan's highest peak.
# 0.946  富士山は日本最高峰の独立峰です。

## Milvus Example

In [ ]:
!pip install "pymilvus[model,milvus_lite]"
# !pip install setuptools

In [ ]:
from pymilvus import MilvusClient
import os, tempfile

model = load_model("ibm-granite/granite-embedding-97m-multilingual-r2",
                   model_kwargs={'dtype': torch.bfloat16, "attn_implementation": "flash_attention_2"})

# Use "./milvus.db" for local persistence or a server URI for production
tmp_dir = tempfile.mkdtemp()
db_path = os.path.join(tmp_dir, "/tmp/milvus.db")
client = MilvusClient(uri=db_path)
client.create_collection(collection_name="multilingual_docs", dimension=384)

docs = [
    "富士山は日本最高峰の独立峰です。",
    "Mount Fuji is Japan's highest peak.",
    "Achy Breaky Heart is a country song written by Don Von Tress.",
    "Berlin ist die Hauptstadt und ein Land der Bundesrepublik Deutschland.",
]
embeddings = model.encode(docs).tolist()
client.insert(
    collection_name="multilingual_docs",
    data=[{"id": i, "vector": emb, "text": doc} for i, (emb, doc) in enumerate(zip(embeddings, docs))],
)

query_emb = model.encode(["What is Japan's tallest mountain?"]).tolist()
results = client.search(
    collection_name="multilingual_docs",
    data=query_emb,
    limit=2,
    output_fields=["text"],
)
for hit in results[0]:
    print(f"{hit['distance']:.3f}  {hit['entity']['text']}")

client.close()
import shutil
shutil.rmtree(tmp_dir)
# 0.961  Mount Fuji is Japan's highest peak.
# 0.913  富士山は日本最高峰の独立峰です

### 4. Framework Integrations

Both models are compatible as drop-in replacements in any framework that accepts Hugging Face embeddings. All three examples below use `MODEL_PATH` from the first cell — change `MODEL_SIZE` there to switch models.

## 5. ONNX Backend

Both R2 models ship with pre-converted ONNX weights. Pass `backend="onnx"` to `SentenceTransformer` — works with any ONNX Runtime backend (CPU, CUDA, TensorRT, DirectML).

> **Note:** Installing `sentence-transformers[onnx]` pulls in `optimum-intel`, which (as of v1.27.0) pins `transformers>=4.45,<4.58` — effectively excluding transformers 5.x (the 4.x line ended at 4.57.6, then jumped to 5.0). This will **downgrade** your transformers if you have 5.x installed. A draft PR ([#1684](https://github.com/huggingface/optimum-intel/pull/1684)) is in progress to support transformers 5.x but has not been released yet. Run the ONNX and OpenVINO sections in a separate environment if you need transformers 5.x for other cells.

Load the pre-built ONNX weights by passing `backend="onnx"` — identical API as the default backend, optimized inference via ONNX Runtime.

In [ ]:
# install onnx support (upgrade optimum to match your transformers version)
!pip install "sentence-transformers[onnx]" "optimum[onnxruntime]" --upgrade

In [ ]:
from sentence_transformers import SentenceTransformer

# ONNX backend uses a different runtime, so we load directly (not cached)
model = SentenceTransformer(MODEL_PATH, backend="onnx", model_kwargs={"file_name": "onnx/model.onnx"})
embeddings = model.encode(["example text"])
print(embeddings.shape)

## 6. OpenVINO Backend (INT8 quantized)

Pre-converted INT8-quantized OpenVINO weights are included; smaller and faster on Intel CPUs.

> **Note:** Same `optimum-intel` version constraint as the ONNX section above — `transformers<4.58` is required until a new release lands. See [optimum-intel#1684](https://github.com/huggingface/optimum-intel/pull/1684).

In [ ]:
!pip install "sentence-transformers[openvino]"

In [ ]:
from sentence_transformers import SentenceTransformer

# OpenVINO INT8-quantized — smaller & faster on CPU
# Note: only the quantized variant is shipped in the repo
model_ov = SentenceTransformer(
    MODEL_PATH,
    backend="openvino",
    model_kwargs={"file_name": "openvino/openvino_model_qint8_quantized.xml"},
)
embeddings_ov = model_ov.encode(["example text"])
print("openvino int8:", embeddings_ov.shape)

## 7. Serving with vLLM

Either model can be served as an embedding endpoint via [vLLM](https://docs.vllm.ai/). Run this in a terminal — it's a long-running server, not a notebook cell.

In [ ]:
# Run in a terminal — this is a server command, not a one-shot notebook call.
# !vllm serve {MODEL_PATH} --task embed

## 8. Convert to GGUF for llama.cpp

Both models can be converted to GGUF for use with [llama.cpp](https://github.com/ggerganov/llama.cpp). Note: Ollama does not currently support ModernBERT-based models.

In [ ]:
# Shell example — convert and embed via llama.cpp
import os
model_short = MODEL_PATH.split("/")[-1]
print(f"# Convert to GGUF")
print(f"python convert_hf_to_gguf.py {MODEL_PATH} \\")
print(f"    --outfile {model_short}.gguf")
print()
print(f"# Generate embeddings")
print(f"llama-embedding -m {model_short}.gguf -p \"example text\"")